In [ ]:
# @title Código de los capítulos anteriores (ejecutar primero)
import math

# Ejemplo A. Unidades: kN y m
nodos = {
    1: {"x": 0.0, "y": 0.0},
    2: {"x": 6.0, "y": 0.0},
    3: {"x": 3.0, "y": 4.0},
}

# E en kN/m² (acero) y área en m²
barras = {
    1: {"nodo_inicial": 1, "nodo_final": 2,
        "elasticidad": 2.0e8, "area": 0.0020},
    2: {"nodo_inicial": 2, "nodo_final": 3,
        "elasticidad": 2.0e8, "area": 0.0020},
    3: {"nodo_inicial": 1, "nodo_final": 3,
        "elasticidad": 2.0e8, "area": 0.0020},
}

# True: el desplazamiento en esa dirección está impedido
apoyos = {
    1: {"x": True, "y": True},   # articulado
    2: {"x": False, "y": True},  # rodillo
}

# Cargas en kN, positivas en el sentido de los ejes
cargas = {
    3: {"x": 20.0, "y": -50.0},
}

import matplotlib.pyplot as plt

# Ejemplo B. Unidades: kN y m
nodos_b = {
    1: {"x": 0.0, "y": 0.0},
    2: {"x": 3.0, "y": 0.0},
    3: {"x": 0.0, "y": 4.0},
    4: {"x": 3.0, "y": 4.0},
}

barras_b = {
    1: {"nodo_inicial": 1, "nodo_final": 2,
        "elasticidad": 2.0e8, "area": 0.0020},
    2: {"nodo_inicial": 2, "nodo_final": 4,
        "elasticidad": 2.0e8, "area": 0.0020},
    3: {"nodo_inicial": 3, "nodo_final": 4,
        "elasticidad": 2.0e8, "area": 0.0020},
    4: {"nodo_inicial": 1, "nodo_final": 3,
        "elasticidad": 2.0e8, "area": 0.0020},
    5: {"nodo_inicial": 1, "nodo_final": 4,
        "elasticidad": 2.0e8, "area": 0.0020},
    6: {"nodo_inicial": 2, "nodo_final": 3,
        "elasticidad": 2.0e8, "area": 0.0020},
}

apoyos_b = {
    1: {"x": True, "y": True},   # articulado
    2: {"x": False, "y": True},  # rodillo
}

cargas_b = {
    3: {"x": 30.0, "y": -40.0},
    4: {"x": 0.0, "y": -40.0},
}

geometria = {}
for numero, barra in barras.items():
    inicial = nodos[barra["nodo_inicial"]]
    final = nodos[barra["nodo_final"]]
    dx = final["x"] - inicial["x"]
    dy = final["y"] - inicial["y"]
    L = math.sqrt(dx**2 + dy**2)
    geometria[numero] = {"longitud": L,
                         "c": dx / L, "s": dy / L}

def calcular_geometria(nodos, barras):
    """Longitud y cosenos directores de cada barra.

    Devuelve un diccionario {número de barra:
    {"longitud": L, "c": c, "s": s}}.
    """
    geometria = {}
    for numero, barra in barras.items():
        inicial = nodos[barra["nodo_inicial"]]
        final = nodos[barra["nodo_final"]]
        dx = final["x"] - inicial["x"]
        dy = final["y"] - inicial["y"]
        L = math.sqrt(dx**2 + dy**2)
        geometria[numero] = {"longitud": L,
                             "c": dx / L, "s": dy / L}
    return geometria

import numpy as np

np.set_printoptions(precision=1, suppress=True)

def rigidez_local(E, A, L):
    """Matriz de rigidez de la barra en sus ejes (2x2)."""
    return E * A / L * np.array([[1.0, -1.0],
                                 [-1.0, 1.0]])


def matriz_transformacion(c, s):
    """Matriz T (2 x 4) tal que u' = T u."""
    return np.array([[c, s, 0.0, 0.0],
                     [0.0, 0.0, c, s]])


def rigidez_barra(E, A, L, c, s):
    """Matriz de rigidez de la barra en ejes globales."""
    T = matriz_transformacion(c, s)
    return T.T @ rigidez_local(E, A, L) @ T

def calcular_rigideces(barras, geometria):
    """Matriz de rigidez global (4 x 4) de cada barra."""
    rigideces = {}
    for numero, barra in barras.items():
        g = geometria[numero]
        rigideces[numero] = rigidez_barra(
            barra["elasticidad"], barra["area"],
            g["longitud"], g["c"], g["s"])
    return rigideces


geometria = calcular_geometria(nodos, barras)
rigideces = calcular_rigideces(barras, geometria)

# Capítulo 6. Los grados de libertad

Este notebook acompaña al capítulo 6 del libro *Análisis matricial de
armaduras con Python*. Numeramos los grados de libertad, los
clasificamos en libres y restringidos, y revisamos la estabilidad.

## 6.2 Decisiones con `if`

In [ ]:
area = barras[1]["area"]
print(area > 0.001, len(barras) == 3, len(nodos) != 3)

In [ ]:
apoyo = apoyos[2]
if apoyo["x"]:
    print("El nodo 2 no puede moverse en x")
else:
    print("El nodo 2 puede moverse en x")

## 6.3 La numeración de los grados de libertad

In [ ]:
def gdl_del_nodo(nodo):
    """Grados de libertad (x, y) del nodo, desde 1."""
    return 2 * nodo - 1, 2 * nodo

In [ ]:
gdl_x, gdl_y = gdl_del_nodo(3)
print(f"Nodo 3: x → {gdl_x}, y → {gdl_y}")

In [ ]:
def gdl_de_barra(barra):
    """Grados de libertad de los extremos de una barra."""
    xi, yi = gdl_del_nodo(barra["nodo_inicial"])
    xj, yj = gdl_del_nodo(barra["nodo_final"])
    return [xi, yi, xj, yj]

In [ ]:
for numero, barra in barras.items():
    print(f"Barra {numero}: {gdl_de_barra(barra)}")

## 6.4 Grados de libertad libres y restringidos

In [ ]:
def clasificar_gdl(nodos, apoyos):
    """Grados de libertad libres y restringidos."""
    libres = []
    restringidos = []
    sin_apoyo = {"x": False, "y": False}
    for nodo in nodos:
        gdl_x, gdl_y = gdl_del_nodo(nodo)
        apoyo = apoyos.get(nodo, sin_apoyo)
        if apoyo["x"]:
            restringidos.append(gdl_x)
        else:
            libres.append(gdl_x)
        if apoyo["y"]:
            restringidos.append(gdl_y)
        else:
            libres.append(gdl_y)
    return libres, restringidos

In [ ]:
libres, restringidos = clasificar_gdl(nodos, apoyos)

In [ ]:
print(f"Libres:       {libres}")
print(f"Restringidos: {restringidos}")

## 6.5 De la numeración desde 1 a los índices de Python

In [ ]:
def indices(gdl):
    """Traduce grados de libertad (desde 1) a índices
    de NumPy (desde 0). Es el único lugar del libro donde
    se hace esta traducción."""
    resultado = []
    for g in gdl:
        resultado.append(g - 1)
    return resultado

In [ ]:
# Un valor por cada grado de libertad del ejemplo A
valores = np.array([10.0, 20.0, 30.0, 40.0, 50.0, 60.0])
print(indices(libres))
valores[indices(libres)]

## 6.6 Una revisión previa de la estabilidad

In [ ]:
def indeterminacion(nodos, barras, restringidos):
    """b + r - 2n: incógnitas menos ecuaciones."""
    b, r, n = len(barras), len(restringidos), len(nodos)
    return b + r - 2 * n

In [ ]:
grado = indeterminacion(nodos, barras, restringidos)
if grado < 0:
    print(f"Inestable: faltan {-grado} barras o apoyos")
elif grado == 0:
    print("Estáticamente determinada (isostática)")
else:
    print(f"Hiperestática de grado {grado}")

## Problema resuelto 6.1: los grados de libertad del ejemplo B

In [ ]:
for numero, barra in barras_b.items():
    print(f"Barra {numero}: {gdl_de_barra(barra)}")

libres_b, restringidos_b = clasificar_gdl(
    nodos_b, apoyos_b)
print(f"Libres:       {libres_b}")
print(f"Restringidos: {restringidos_b}")

grado_b = indeterminacion(nodos_b, barras_b,
                          restringidos_b)
print(f"Grado de indeterminación: {grado_b}")

## Ejercicios

Resuelva cada ejercicio en su notebook y después ejecute la celda de
verificación correspondiente.

In [ ]:
# Ejercicio 6.1: grados de libertad de la barra 6 del ejemplo B
respuesta = None  # reemplace None por una lista, por ejemplo [1, 2, 3, 4]

if respuesta is None:
    print("Escriba su resultado en lugar de None y vuelva a ejecutar la celda.")
elif respuesta == [3, 4, 5, 6]:
    print("Correcto.")
else:
    print("Todavía no. La barra 6 va del nodo 2 al nodo 3.")

In [ ]:
# Ejercicio 6.2: grado de indeterminación con dos apoyos articulados
respuesta = None  # reemplace None por su resultado

if respuesta is None:
    print("Escriba su resultado en lugar de None y vuelva a ejecutar la celda.")
elif respuesta == 1:
    print("Correcto.")
else:
    print("Todavía no. ¿Cuántas reacciones tiene ahora la armadura?")